# Steps 7 & 8: Anomaly Segmentation Baselines

- Step 7: pixel-based baselines using ERFNet (MSP, MaxLogit, Max Entropy)
- Step 8: mask-based baselines using EoMT (MSP, MaxLogit, Max Entropy, RbA) evaluated on three checkpoints + temperature scaling

In [1]:
!pip install ood_metrics >/dev/null
!pip install -U 'jsonargparse[signatures]>=4.27.7' >/dev/null
!pip install gitignore_parser > /dev/null
!pip install lightning > /dev/null
!pip install 'torchao>=0.16.0' >/dev/null

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_v

## Setup and Imports

In [2]:
import os
import sys
import torch
from tqdm.notebook import tqdm
from google.colab import drive

# Uninstall conflicting numpy and pandas versions, then reinstall to ensure compatibility
!pip uninstall -y numpy pandas >/dev/null
!pip install numpy==2.0.0 >/dev/null
!pip install pandas >/dev/null

# Import numpy and pandas *after* ensuring the correct versions are installed
import numpy as np
import pandas as pd

drive.mount('/content/drive', force_remount=True);

PROJECT_ROOT = '/content/drive/MyDrive/FundGitHubProject/'
for path in [PROJECT_ROOT, os.path.join(PROJECT_ROOT, 'eomt')]:
    if path not in sys.path:
        sys.path.insert(0, path)

if not os.path.exists('/content/Fundamental_Project'):
    os.symlink(PROJECT_ROOT, '/content/Fundamental_Project')

os.chdir('/content/Fundamental_Project')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cufflinks 0.17.3 requires pandas>=0.19.2, which is not installed.
seaborn 0.13.2 requires pandas>=1.2, which is not installed.
spreg 1.9.0 requires pandas, which is not installed.
shap 0.51.0 requires pandas, which is not installed.
holoviews 1.22.1 requires pandas>=1.3, which is not installed.
arviz 0.22.0 requires pandas>=2.1.0, which is not installed.
mlxtend 0.23.4 requires pandas>=0.24.2, which is not installed.
esda 2.9.0 requires pandas>=2.1, which is not installed.
pysal 25.7 requires pandas>=1.4, which is not installed.
access 1.1.10.post3 requires pandas>=2.1.0, which is not installed.
bigframes 2.40.0 requires pandas>=1.5.3, which is not installed.
gradio 5.50.0 requires pandas<3.0,>=1.0, which is not installed.
yfinance 0.2.66 requires pandas>=1.3.0, which is not installed.
bqplot 0.12.47 requires pand

In [3]:
# Scoring functions and metric computation from the project repo
from posthoc_metrics import (
    get_pixel_msp,
    get_pixel_max_logit,
    get_pixel_entropy,
    get_mask_msp,
    get_mask_max_logit,
    get_mask_entropy,
    get_mask_rba,
    compute_metrics,
    cache_model_outputs,
    fast_temperature_search,
)

import torch.nn.functional as F
from eval.Validation_Dataset import anomaly_datasets
from eval.erfnet import ERFNet
from eomt.models.vit import ViT
from eomt.models.eomt import EoMT
from eomt.training.mask_classification_semantic import MaskClassificationSemantic
from eomt.training.mask_classification_panoptic import MaskClassificationPanoptic

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


## Load datasets
Same five benchmarks used for both Step 7 and Step 8.
Loaded once and reused across all models and methods.


In [5]:
DATASETS = {
    'SMIYC RA-21':  'RoadAnomaly21',
    'SMIYC RO-21':  'RoadObstacle21',
    'FS L&F':       'FS_LostFound',
    'FS Static':    'FS_Static',
    'Road Anomaly': 'RoadAnomaly',
}

dataloaders = {}
for display_name, internal_name in DATASETS.items():
    dm = anomaly_datasets.AnomalyDataModule(
        dataset_name=internal_name,
        img_size=(640, 640)
    )
    dm.setup()
    dataloaders[display_name] = dm.val_dataloader()
    print(f"Loaded: {display_name}")

Loaded: SMIYC RA-21
Loaded: SMIYC RO-21
Loaded: FS L&F
Loaded: FS Static
Loaded: Road Anomaly


In [6]:
# ImageNet stats used only for EoMT (DINOv2 backbone)
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

In [9]:
def evaluate(model, dataloader, scoring_fn, desc="",
             input_scale=1.0, normalize=False, debug=False):
    """
    Run inference and compute AUPRC / FPR95 for anomaly segmentation.

    Args
    ----
    model        : segmentation model already on `device`
    dataloader   : yields {'image': (B,3,H,W), 'label': (B,H,W)}
                   images ∈ [0,1] as returned by the anomaly dataloaders
    scoring_fn   : model_output → (B,H,W)  anomaly score, **higher = more anomalous**
    input_scale  : pre-multiply images by this scalar before forwarding
                   • ERFNet : 255.0  (trained on uint8 range)
                   • EoMT   : 1.0    (after ImageNet norm stays ≈ [-2.5, 2.5])
    normalize    : apply ImageNet mean/std norm (True for EoMT)
    debug        : print score gap on first batch
    """
    model.eval()
    all_scores, all_labels = [], []

    with torch.no_grad():
        for i, batch in enumerate(tqdm(dataloader, desc=desc, leave=False)):
            images = batch['image'].to(device) * input_scale   # ← BUG FIX #1 applied via caller
            labels = batch['label']                             # (B,H,W) CPU; 255 = ignore

            if normalize:
                images = (images - IMAGENET_MEAN) / IMAGENET_STD

            output = model(images)
            scores = scoring_fn(output)      # (B,H,W)

            # Resize scores to GT resolution if model output is downsampled
            if scores.shape[-2:] != labels.shape[-2:]:
                scores = F.interpolate(
                    scores.unsqueeze(1).float(),
                    size=labels.shape[-2:],
                    mode='bilinear',
                    align_corners=False,
                ).squeeze(1)

            # ── debug: print score gap (anomaly mean > normal mean → correct polarity)
            if debug and i == 0:
                s = scores.cpu()
                valid        = labels != 255
                normal_mask  = valid & (labels == 0)
                anomaly_mask = valid & (labels == 1)
                print(f"\n[debug] score range : [{s.min():.4f}, {s.max():.4f}]")
                if normal_mask.any() and anomaly_mask.any():
                    print(f"[debug] normal  mean : {s[normal_mask].mean():.4f}")
                    print(f"[debug] anomaly mean : {s[anomaly_mask].mean():.4f}")
                    gap = s[anomaly_mask].mean() - s[normal_mask].mean()
                    print(f"[debug] gap          : {gap:.4f}  (should be > 0)")
                else:
                    print("[debug] first batch has no anomaly pixels — try a different dataset")

            valid = (labels != 255)

            # ← BUG FIX #12: scores.cpu() called correctly once here
            all_scores.append(scores.cpu().numpy()[valid.numpy()])
            all_labels.append(labels.numpy()[valid.numpy()])

    return compute_metrics(
        np.concatenate(all_scores),
        np.concatenate(all_labels),
    )


def run_evaluation(model, model_name, scoring_methods, dataloaders,
                   input_scale=1.0, normalize=False, debug=False):
    rows = []
    for method_name, scoring_fn in scoring_methods.items():
        row = {'Model': model_name, 'Method': method_name}
        for ds_name, loader in dataloaders.items():
            metrics = evaluate(
                model, loader, scoring_fn,
                desc=f"{model_name} | {method_name} | {ds_name}",
                input_scale=input_scale,
                normalize=normalize,
                debug=(debug and ds_name == 'SMIYC RA-21' and method_name == 'MSP'),
            )
            row[f"{ds_name} AuPRC"] = round(metrics['auprc'] * 100, 2)
            row[f"{ds_name} FPR95"] = round(metrics['fpr95'] * 100, 2)
        rows.append(row)
        print(f"  {method_name} done")
    return rows

# Step 7: Pixel-based Baselines (ERFNet)
Evaluating ERFNet with pixel-wise scoring methods.

## Load ERFNet

In [10]:
# Load ERFNet
erfnet = ERFNet(num_classes=20).to(device)

weights_path = 'trained_models/erfnet_pretrained.pth'
assert os.path.exists(weights_path), f"Weights not found at {weights_path}"
checkpoint  = torch.load(weights_path, map_location=device)
state_dict  = checkpoint.get('state_dict', checkpoint)
# Strip DataParallel 'module.' prefix if present
state_dict  = {
    (k[len('module.'):] if k.startswith('module.') else k): v
    for k, v in state_dict.items()
}
missing, unexpected = erfnet.load_state_dict(state_dict, strict=False)
print(f"ERFNet loaded — missing: {len(missing)}, unexpected: {len(unexpected)}")

ERFNet loaded — missing: 2, unexpected: 0


In [11]:
erfnet.eval()

ERFNet(
  (encoder): Encoder(
    (initial_block): DownsamplerBlock(
      (conv): Conv2d(3, 13, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    )
    (layers): ModuleList(
      (0): DownsamplerBlock(
        (conv): Conv2d(16, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1-5): 5 x non_bottleneck_1d(
        (conv3x1_1): Conv2d(64, 64, kernel_size=(3, 1), stride=(1, 1), padding=(1, 0))
        (conv1x3_1): Conv2d(64, 64, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1))
        (bn1): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
        (conv3x1_2): Conv2d(64

In [12]:
def pixel_scoring_methods_from_logits(logits):
    # logits: (B, 20, H, W)
    # drop the void class before computing scores
    sem_logits = logits[:, 1:, :, :]   # (B, 19, H, W) — adjust index if void is last
    return sem_logits

pixel_scoring_methods = {
    'MSP':         get_pixel_msp,        # already returns 1 - max_prob (high = anomaly)
    'MaxLogit':    get_pixel_max_logit,  # already returns -max_logit  (high = anomaly)
    'Max Entropy': get_pixel_entropy,    # already returns normalized H (high = anomaly)
}

In [13]:
# Run one batch and see where the model puts high probability on known-void regions
checkpoint = torch.load(weights_path, map_location='cpu')
# print the keys to see if there's a class list stored
print([k for k in checkpoint.keys()])

['module.encoder.initial_block.conv.weight', 'module.encoder.initial_block.conv.bias', 'module.encoder.initial_block.bn.weight', 'module.encoder.initial_block.bn.bias', 'module.encoder.initial_block.bn.running_mean', 'module.encoder.initial_block.bn.running_var', 'module.encoder.layers.0.conv.weight', 'module.encoder.layers.0.conv.bias', 'module.encoder.layers.0.bn.weight', 'module.encoder.layers.0.bn.bias', 'module.encoder.layers.0.bn.running_mean', 'module.encoder.layers.0.bn.running_var', 'module.encoder.layers.1.conv3x1_1.weight', 'module.encoder.layers.1.conv3x1_1.bias', 'module.encoder.layers.1.conv1x3_1.weight', 'module.encoder.layers.1.conv1x3_1.bias', 'module.encoder.layers.1.conv3x1_2.weight', 'module.encoder.layers.1.conv3x1_2.bias', 'module.encoder.layers.1.conv1x3_2.weight', 'module.encoder.layers.1.conv1x3_2.bias', 'module.encoder.layers.1.bn1.weight', 'module.encoder.layers.1.bn1.bias', 'module.encoder.layers.1.bn1.running_mean', 'module.encoder.layers.1.bn1.running_var'

In [14]:
pixel_results = run_evaluation(
    erfnet, 'ERFNet', pixel_scoring_methods, dataloaders,
    input_scale=255.0,
    normalize=False,   # ← correct for this checkpoint
    debug=True,
)

ERFNet | MSP | SMIYC RA-21:   0%|          | 0/10 [00:00<?, ?it/s]


[debug] score range : [0.0000, 0.6611]
[debug] normal  mean : 0.0127
[debug] anomaly mean : 0.0170
[debug] gap          : 0.0043  (should be > 0)


ERFNet | MSP | SMIYC RO-21:   0%|          | 0/30 [00:00<?, ?it/s]

ERFNet | MSP | FS L&F:   0%|          | 0/100 [00:00<?, ?it/s]

ERFNet | MSP | FS Static:   0%|          | 0/30 [00:00<?, ?it/s]

...

ERFNet | Max Entropy | SMIYC RO-21:   0%|          | 0/30 [00:00<?, ?it/s]

ERFNet | Max Entropy | FS L&F:   0%|          | 0/100 [00:00<?, ?it/s]

ERFNet | Max Entropy | FS Static:   0%|          | 0/30 [00:00<?, ?it/s]

ERFNet | Max Entropy | Road Anomaly:   0%|          | 0/60 [00:00<?, ?it/s]

  Max Entropy done


In [15]:
df_pixel = pd.DataFrame(pixel_results)
print("\nStep 7 — ERFNet results:")
print(df_pixel.to_string(index=False))


Step 7 — ERFNet results:
 Model      Method  SMIYC RA-21 AuPRC  SMIYC RA-21 FPR95  SMIYC RO-21 AuPRC  SMIYC RO-21 FPR95  FS L&F AuPRC  FS L&F FPR95  FS Static AuPRC  FS Static FPR95  Road Anomaly AuPRC  Road Anomaly FPR95
ERFNet         MSP              85.37              100.0              99.30              100.0         99.74        100.00            98.81           100.00               90.56              100.00
ERFNet    MaxLogit              86.45               89.1              99.42               92.7         99.81         90.02            99.19            90.76               91.30               90.87
ERFNet Max Entropy              85.46              100.0              99.26              100.0         99.75        100.00            98.86           100.00               90.75              100.00


# Step 8: Mask-based Baselines (EoMT)
Evaluating EoMT across three checkpoints with query-based and RbA scoring.

In [ ]:
def make_mask_scoring_fn(fn):
    def wrapped(output):
        mask_pred_list, class_pred_list = output      # masks first, classes second
        mask_pred  = mask_pred_list[-1]  if isinstance(mask_pred_list,  list) else mask_pred_list
        class_pred = class_pred_list[-1] if isinstance(class_pred_list, list) else class_pred_list
        # mask_pred:  (B, Q, H, W)        spatial
        # class_pred: (B, Q, num_classes)  semantic
        return fn(class_pred, mask_pred)
    return wrapped

In [ ]:
def neg_mask(fn):
    """Negate a mask scoring function's output."""
    def wrapper(output):
        return -make_mask_scoring_fn(fn)(output)
    return wrapper

In [ ]:
mask_scoring_methods = {
    'MSP':         make_mask_scoring_fn(get_mask_msp),
    'MaxLogit':    make_mask_scoring_fn(get_mask_max_logit),
    'Max Entropy': make_mask_scoring_fn(get_mask_entropy),   # already correct polarity
    'RbA':         make_mask_scoring_fn(get_mask_rba),       # already correct polarity
}

In [ ]:
mask_scoring_methods = {
        'MSP': get_mask_msp,
        'MaxLogit': lambda m_cls, m_pred: get_mask_max_logit(m_cls, m_pred),
        'Max Entropy': get_mask_entropy,
        'RbA': get_mask_rba
    }

In [ ]:
# Three checkpoints: fine-tuned (COCO -> Cityscapes via LoRA)
CHECKPOINTS = {
    'Fine-tuned': 'checkpoints/cityscapes_enhanced/eomt-enhanced-epoch=23-val_iou_all=0.00.ckpt',
}

In [ ]:
import torch.nn.functional as F

def _load_state_dict_into(model, ckpt_path):
    state = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    if isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']
    if not any(k.startswith('network.') for k in state):
        state = {f'network.{k}': v for k, v in state.items()}

    model_sd = model.state_dict()
    for key in list(state.keys()):
        if 'pos_embed' not in key:
            continue
        if key not in model_sd:
            continue
        ckpt_shape  = state[key].shape
        model_shape = model_sd[key].shape
        if ckpt_shape == model_shape:
            continue

        N_ckpt, D   = ckpt_shape[1], ckpt_shape[2]
        N_model     = model_shape[1]
        H_c = W_c   = int(N_ckpt  ** 0.5)
        H_m = W_m   = int(N_model ** 0.5)

        pe = state[key]
        pe = pe.reshape(1, H_c, W_c, D).permute(0, 3, 1, 2)
        pe = F.interpolate(pe.float(), size=(H_m, W_m),
                           mode='bicubic', align_corners=False)
        pe = pe.permute(0, 2, 3, 1).reshape(1, N_model, D)
        state[key] = pe
        print(f'  interpolated {key}: {list(ckpt_shape)} -> {list(model_shape)}')

    model.load_state_dict(state, strict=False)
    print(f'  loaded {ckpt_path}')

    return model

def load_cs_model(ckpt_path, img_size=(1024, 1024)):
    encoder = ViT(img_size=img_size, backbone_name='vit_base_patch14_reg4_dinov2')
    network = EoMT(encoder, num_classes=19, num_q=100, num_blocks=3)
    model = MaskClassificationSemantic(
        network=network, img_size=img_size, num_classes=19, attn_mask_annealing_enabled=False
    )
    return _load_state_dict_into(model, ckpt_path).eval().to(device)

def load_coco_model(ckpt_path, img_size=(640, 640)):
    stuff_classes = list(range(80, 133))
    encoder = ViT(img_size=img_size, backbone_name='vit_base_patch14_reg4_dinov2')
    network = EoMT(encoder, num_classes=133, num_q=200, num_blocks=3)
    model = MaskClassificationPanoptic(
        network=network, img_size=img_size, num_classes=133,
        stuff_classes=stuff_classes, attn_mask_annealing_enabled=False
    )
    return _load_state_dict_into(model, ckpt_path).eval().to(device)

In [ ]:
print('Loading EoMT-Cityscapes...')
model_cs = load_cs_model("eomt/eomt_weights/eomt_cityscapes.bin", img_size=(640, 640))

In [ ]:
print('\nLoading EoMT-COCO...')
model_coco = load_coco_model("eomt/eomt_weights/eomt_coco.bin", img_size=(640, 640))

In [ ]:
# Models to evaluate and their display names for the results table
eomt_models = [
    (model_cs,   'EoMT (Cityscapes)'),
    (model_coco, 'EoMT (COCO)'),
]

In [ ]:
mask_results = []
for model, model_name in eomt_models:
    rows = run_evaluation(
        model, model_name, mask_scoring_methods, dataloaders,
        input_scale=1.0,   # images already in [0,1]; ImageNet norm applied below
        normalize=True,    # DINOv2 backbone expects ImageNet-normalised input
        debug=True,
    )
    mask_results.extend(rows)
    torch.cuda.empty_cache()

df_mask = pd.DataFrame(mask_results)
print("\nStep 8 — EoMT results:")
print(df_mask.to_string(index=False))

In [ ]:
from eval.Validation_Dataset import anomaly_datasets
from eomt.checkpoint_utils import get_finetuned_model
def evaluate_mask_model(model, dataloader, method_name):
    model.eval()
    all_scores, all_gts = [], []

    scoring_fns = {
        'MSP': get_mask_msp,
        'MaxLogit': lambda m_cls, m_pred: get_mask_max_logit(m_cls, m_pred),
        'Max Entropy': get_mask_entropy,
        'RbA': get_mask_rba
    }
    fn = scoring_fns[method_name]

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"EoMT {method_name}"):
            img, gt = batch['image'].to(device), batch['label']
            m_pred, m_cls = model(img) # EoMT returns (mask_pred_list, class_pred_list)

            # Use last layer outputs
            score_map = fn(m_cls[-1], m_pred[-1])

            if score_map.shape[-2:] != gt.shape[-2:]:
                score_map = torch.nn.functional.interpolate(score_map.unsqueeze(1), size=gt.shape[-2:], mode='bilinear').squeeze(1)

            valid = (gt != 255)
            all_scores.append(score_map.cpu().numpy().flatten())
            all_gts.append(gt.numpy().flatten())

    return compute_metrics(np.concatenate(all_scores), np.concatenate(all_gts))

checkpoints = {
    'COCO': 'checkpoints/eomt_coco.ckpt',
    'Cityscapes': 'checkpoints/eomt_cityscapes.ckpt',
    'Fine-tuned': 'checkpoints/cityscapes_enhanced/eomt-enhanced-epoch=23-val_iou_all=0.00.ckpt'
}

mask_results = []
for ckpt_name, ckpt_path in checkpoints.items():
    if not os.path.exists(ckpt_path): continue
    model = get_finetuned_model(ckpt_path).to(device)

    for method in ['MSP', 'MaxLogit', 'Max Entropy', 'RbA']:
        row = {'Model': f'EoMT ({ckpt_name})', 'Method': method}
        for ds_name, loader in dataloaders.items():
            metrics = evaluate_mask_model(model, loader, method)
            row[f"{ds_name} AuPRC"] = metrics['auprc']
            row[f"{ds_name} FPR95"] = metrics['fpr95']
        mask_results.append(row)

    del model; torch.cuda.empty_cache()

df_mask = pd.DataFrame(mask_results)
df_mask

## Temperature Scaling (Smart Trick)
Optimizing MSP calibration using cached logits.

In [ ]:
# Fix a stale function name in the fast_eval_utils module if needed.
# (The original code patched the source file on disk — that is fragile.
#  A safer approach is to monkey-patch the module in memory.)
import posthoc_metrics.fast_eval_utils as _feu
if hasattr(_feu, 'get_msp_anomaly_map') and not hasattr(_feu, 'get_mask_msp'):
    _feu.get_mask_msp = _feu.get_msp_anomaly_map
    importlib.reload(sys.modules['posthoc_metrics'])

from posthoc_metrics import fast_temperature_search

In [ ]:
# Step 1: run inference once and cache logits
calib_model = get_finetuned_model(CHECKPOINTS['Fine-tuned']).to(device)
calib_model.eval()

CACHE_DIR = 'temp_scaling_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

# The cache function expects (image, label) tuples, not dicts.
class TupleLoader:
    def __init__(self, loader):
        self.loader = loader
    def __iter__(self):
        for batch in self.loader:
            yield batch['image'], batch['label']
    def __len__(self):
        return len(self.loader)

cache_model_outputs(
    calib_model,
    TupleLoader(dataloaders['Road Anomaly']),
    CACHE_DIR,
    device=device,
)
del calib_model
torch.cuda.empty_cache()

In [ ]:
# Step 2: sweep temperatures
# The range here is narrow (0.5 to 1.1). You could widen it if the
# optimum is at the boundary — e.g. try [0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0].

import os
import numpy as np
import torch
from tqdm import tqdm

# Access the original module where fast_temperature_search is defined
import posthoc_metrics.fast_eval_utils as _feu

_original_fast_temperature_search = _feu.fast_temperature_search

def _corrected_fast_temperature_search(cache_dir, scoring_fn, temperatures):
    results = {}
    cached_files = [f for f in os.listdir(cache_dir) if f.endswith('.npz')]
    num_cached_files = len(cached_files)

    for T in temperatures:
        print(f"Testing Temperature T={T}...")
        all_scores = []
        all_gts = []

        for i in tqdm(range(num_cached_files), desc=f"Testing Temperature T={T}"):
            data = np.load(os.path.join(cache_dir, f"{i:04d}.npz"))

            # Convert to tensors and move to device
            class_preds = torch.from_numpy(data['class_preds']).to(device)
            mask_preds = torch.from_numpy(data['mask_preds']).to(device)
            labels = data['labels'] # numpy array, original GT size

            # Apply the scoring function with the current temperature
            # Assuming scoring_fn accepts temperature argument from the library's fast_temperature_search signature
            scores = scoring_fn(class_preds, mask_preds, temperature=T).cpu().numpy()

            # Ensure scores match the labels' spatial dimensions
            if scores.shape[-2:] != labels.shape[-2:]:
                scores_tensor = torch.from_numpy(scores).unsqueeze(1) # Add channel dim for interpolation
                labels_tensor = torch.from_numpy(labels) # For target size

                interpolated_scores = torch.nn.functional.interpolate(
                    scores_tensor,
                    size=labels_tensor.shape[-2:], # Use original label size for interpolation
                    mode='bilinear',
                    align_corners=False,
                ).squeeze(1).numpy() # Remove channel dim and convert back to numpy

                scores = interpolated_scores

            # Create a valid mask to filter out ignore labels (255)
            valid_mask = (labels != 255)

            # Append only valid scores and labels, flattened
            all_scores.append(scores[valid_mask].ravel())
            all_gts.append(labels[valid_mask].ravel())

        # Compute aggregate metrics for this temperature
        # `compute_metrics` is imported from `posthoc_metrics` in setup cell
        metrics = compute_metrics(np.concatenate(all_scores), np.concatenate(all_gts))
        print(f"  T={T} -> AuPRC: {metrics['auprc']:.2f} | FPR95: {metrics['fpr95']:.2f}")
        results[T] = metrics
    return results

# Monkey-patch the function in the imported module
_feu.fast_temperature_search = _corrected_fast_temperature_search

# Now, call the function using the module's reference, which is now patched.
# The fast_temperature_search function imported globally (if any) might still refer to the original.
# To be safe, let's call it via the module, or ensure the global one is updated.
# If `fast_temperature_search` was imported as `from posthoc_metrics import fast_temperature_search`,
# it might hold a reference to the old function. Reloading `posthoc_metrics` would update it.
# However, the instruction is to fix *this cell*, so directly using `_feu.fast_temperature_search` is safest.

# If `fast_temperature_search` (global) is what's expected, we need to update it:
fast_temperature_search = _feu.fast_temperature_search

temperatures = [0.5, 0.75, 1.0, 1.1]
search_results = fast_temperature_search(
    CACHE_DIR,
    scoring_fn=get_mask_msp,
    temperatures=temperatures,
)


In [ ]:
# Step 3: find best T by AuPRC
best_t = max(search_results, key=lambda t: search_results[t]['auprc'])

# Step 4: build results table
temp_rows = []
for t in temperatures:
    temp_rows.append({
        'Temperature': t,
        'AuPRC': round(search_results[t]['auprc'] * 100, 2),
        'FPR95': round(search_results[t]['fpr95'] * 100, 2),
        'Note': 'best' if t == best_t else '',
    })

In [ ]:
df_temp = pd.DataFrame(temp_rows)
print(f"\nTemperature scaling results (MSP, Fine-tuned EoMT, Road Anomaly)")
print(f"Best temperature: T = {best_t}")
print(df_temp.to_string(index=False))